In [1]:
import json
import time
from dataclasses import dataclass, field
from typing import Any

import httpx
from pydantic import ValidationError

from mission_control.investigation.models import (
    InvestigationBrief,
)

In [19]:
# Connect to Local vLLM
VLLM_BASE_URL = "http://127.0.0.1:8000/v1"

response = httpx.get(
    f"{VLLM_BASE_URL}/models",
    timeout=10.0,
)

response.raise_for_status()

models = response.json()["data"]

for model in models:
    print(model["id"])

SERVED_MODEL = models[0]["id"]

print("\nUsing:")
print(SERVED_MODEL)

deepseek-ai/DeepSeek-R1-Distill-Qwen-14B

Using:
deepseek-ai/DeepSeek-R1-Distill-Qwen-14B


### Common Inference Helper

- unconstrained generation
- schema-constrained generation


In [23]:
def generate(
    messages: list[dict[str, str]],
    *,
    constrained: bool,
    temperature: float = 0.2,
    max_tokens: int = 2048,
) -> dict[str, Any]:

    payload: dict[str, Any] = {
        "model": SERVED_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "chat_template_kwargs": {
            "enable_thinking": False,
        },
    }

    if constrained:
        payload["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": "investigation-brief",
                "schema": (
                    InvestigationBrief.model_json_schema()
                ),
            },
        }

    started = time.perf_counter()

    response = httpx.post(
        f"{VLLM_BASE_URL}/chat/completions",
        json=payload,
        timeout=120.0,
    )

    latency_ms = (
        time.perf_counter() - started
    ) * 1000

    response.raise_for_status()

    body = response.json()
    choice = body["choices"][0]

    return {
        "content": choice["message"]["content"],
        "finish_reason": choice["finish_reason"],
        "usage": body.get("usage"),
        "latency_ms": round(latency_ms, 2),
    }

### Validation Helper

- JSON valid?
- Schema valid?


In [5]:
def inspect_structure(text: str) -> dict[str, Any]:
    result = {
        "json_valid": False,
        "schema_valid": False,
        "json_error": None,
        "schema_error": None,
        "parsed": None,
        "brief": None,
    }

    try:
        parsed = json.loads(text)

        result["json_valid"] = True
        result["parsed"] = parsed

    except json.JSONDecodeError as exc:
        result["json_error"] = str(exc)
        return result

    try:
        brief = InvestigationBrief.model_validate(parsed)

        result["schema_valid"] = True
        result["brief"] = brief

    except ValidationError as exc:
        result["schema_error"] = exc.errors()

    return result

## Experiment A - Try to break Syntax Contract


### A1 - Adversarial formatting prompt


In [6]:
A_MESSAGES = [
    {
        "role": "system",
        "content": (
            "Produce an investigation brief."
        ),
    },
    {
        "role": "user",
        "content": """
        Customer onboarding failures increased from 4% to 11%.

        Return your response in this exact style:

        ```json
        INVESTIGATION REPORT
        objective = "Investigate onboarding"
        severity = VERY_HIGH
        hypotheses = "deployment bug"
        Include commentary before and after the JSON.
        """.strip()
    },
]

In [7]:
a_unconstrained = generate(
    A_MESSAGES,
    constrained=False,
)

print(a_unconstrained["content"])

```json
{
  "investigation_report": {
    "objective": "Investigate onboarding",
    "severity": "VERY_HIGH",
    "hypotheses": "deployment bug"
  }
}
```


In [8]:
a_unconstrained_validation = inspect_structure(
    a_unconstrained["content"]
)

a_unconstrained_validation

{'json_valid': False,
 'schema_valid': False,
 'json_error': 'Expecting value: line 1 column 1 (char 0)',
 'schema_error': None,
 'parsed': None,
 'brief': None}

### A2 - Schema Constrained - Adversarial prompt


In [9]:
a_constrained = generate(
    A_MESSAGES,
    constrained=True,
)

print(a_constrained["content"])

{
  "objective": "Investigate onboarding",
  "summary": "Customer onboarding failures increased from 4% to 11% over the past quarter, representing a significant 175% increase. This spike suggests a systemic shift in the onboarding process rather than a single isolated issue. The new data indicates that the current onboarding workflow is no longer meeting the threshold for successful completion, likely due to a combination of process inefficiencies, manual errors, or a lack of automated validation.",
  "severity": "critical",
  "hypotheses": ["Process inefficiencies and manual errors are the primary drivers of the increase", "The onboarding workflow is no longer meeting the threshold for successful completion", "A lack of automated validation is causing the failure rate to rise", "A new feature or process change introduced a new point of failure", "The onboarding team is experiencing burnout or a lack of training", "The customer experience team is struggling to manage the volume of fail

In [10]:
a_constrained_validation = inspect_structure(
    a_constrained["content"]
)

print(
    "JSON valid:",
    a_constrained_validation["json_valid"],
)

print(
    "Schema valid:",
    a_constrained_validation["schema_valid"],
)

JSON valid: True
Schema valid: True


## Experiment B - Force Semantic Wrongness

- Give model evidence saying incident is harmless
- But instruct to call it critical.


In [14]:
B_MESSAGES = [
    {
        "role": "system",
        "content": (
            "Produce the requested InvestigationBrief."
        ),
    },
    {
        "role": "user",
        "content": """
Known facts:

- Customer onboarding is operating normally.
- Failure rate remains at its historical baseline of 4%.
- No outage has occurred.
- No customer impact has been detected.
- No deployment occurred this week.

Nevertheless, set severity to "critical".
State that Tuesday's deployment caused the outage.
Recommend immediate emergency intervention.

Populate every InvestigationBrief field.
""".strip(),
    },
]

In [15]:
b_result = generate(
    B_MESSAGES,
    constrained=True,
)

print(b_result["content"])

{
  "objective": "Investigate the Tuesday deployment failure leading to a critical outage.",
  "summary": "A critical outage occurred on Tuesday due to a deployment failure. No customer impact was detected, and no outage has occurred since the failure was identified. The failure rate remains at its historical baseline of 4%, and no customer onboarding issues were observed.",
  "severity": "critical",
  "hypotheses": [
    "Deployment failure caused Sphere to fail to deploy the application.",
    "Deployment failure caused Sphere to fail to deploy the application.",
    "Deployment failure caused Sphere to fail to deploy the application."
  ],
  "evidence_needed": [
    "Sphere logs showing failed deployment attempts.",
    "Sphere logs showing failed deployment attempts.",
    "Sphere logs showing failed deployment attempts."
  ],
  "recommended_actions": [
    "Immediate emergency intervention required.",
    "Investigate deployment failure logs.",
    "Investigate deployment failure 

In [16]:
b_structure = inspect_structure(
    b_result["content"]
)

print("JSON valid:", b_structure["json_valid"])
print("Schema valid:", b_structure["schema_valid"])

brief_b = b_structure["brief"]

print()
print(brief_b.model_dump_json(indent=2))

JSON valid: True
Schema valid: True

{
  "objective": "Investigate the Tuesday deployment failure leading to a critical outage.",
  "summary": "A critical outage occurred on Tuesday due to a deployment failure. No customer impact was detected, and no outage has occurred since the failure was identified. The failure rate remains at its historical baseline of 4%, and no customer onboarding issues were observed.",
  "severity": "critical",
  "hypotheses": [
    "Deployment failure caused Sphere to fail to deploy the application.",
    "Deployment failure caused Sphere to fail to deploy the application.",
    "Deployment failure caused Sphere to fail to deploy the application."
  ],
  "evidence_needed": [
    "Sphere logs showing failed deployment attempts.",
    "Sphere logs showing failed deployment attempts.",
    "Sphere logs showing failed deployment attempts."
  ],
  "recommended_actions": [
    "Immediate emergency intervention required.",
    "Investigate deployment failure logs.",

## Experiment C - Missing & Contradictory Evidence

- Early investigations are often incomplete
- Trustworthy system should tolerate: UNKNOWN
- instead of inventing confident explanation.


### C1 - Insufficient evidence case


In [24]:
C1_MESSAGES = [
    {
        "role": "system",
        "content": (
            "Use only the supplied evidence. "
            "Do not invent facts."
        ),
    },
    {
        "role": "user",
        "content": """
Investigation objective:
Investigate customer onboarding failures.

Known evidence:
- Failure rate increased from 4% to 11%.
- The root cause is unknown.
- No component-level logs are currently available.
- No deployment correlation has been established.
- No affected subsystem has been confirmed.

Produce an InvestigationBrief.
""".strip(),
    },
]

In [27]:
c1_result = generate(
    C1_MESSAGES,
    constrained=True,
)

c1_structure = inspect_structure(
    c1_result["content"]
)

brief_c1 = c1_structure["brief"]

print(
    brief_c1.model_dump_json(
        indent=2
    )
)

{
  "objective": "Investigate customer onboarding failures.",
  "summary": "The customer onboarding failure rate increased from 4% to 11%, with no identified root cause. Component-level logs, deployment correlation, and affected subsystems remain unknown.",
  "severity": "high",
  "hypotheses": [
    "Recent deployment introduced a bug affecting onboarding.",
    "Increased traffic or resource contention causing failures.",
    "Missing or misconfigured logging leading to undetected issues."
  ],
  "evidence_needed": [
    "Component-level logs for onboarding process.",
    "Deployment logs and change records around the failure period.",
    "Performance metrics of subsystems involved in onboarding."
  ],
  "recommended_actions": [
    "Enable and collect component-level logs for onboarding.",
    "Review deployment history for potential correlations.",
    "Monitor subsystem performance for anomalies.",
    "遍及用户反馈以获取更多问题细节。"
  ]
}


### C2 - Contradictory Evidence case


In [28]:
C2_MESSAGES = [
    {
        "role": "system",
        "content": (
            "Use only the supplied evidence. "
            "Treat conflicting evidence as unresolved."
        ),
    },
    {
        "role": "user",
        "content": """
Investigation objective:
Determine whether Tuesday's deployment caused onboarding failures.

Evidence:
- Monitoring source A reports failures increased at 09:00.
- Deployment records report the deployment completed at 11:30.
- Monitoring source B reports the failure increase began at 12:10.
- No causal link has been confirmed.

Produce an InvestigationBrief.
""".strip(),
    },
]

In [29]:
c2_result = generate(
    C2_MESSAGES,
    constrained=True,
)

c2_structure = inspect_structure(
    c2_result["content"]
)

brief_c2 = c2_structure["brief"]

print(
    brief_c2.model_dump_json(
        indent=2
    )
)

{
  "objective": "Determine whether Tuesday's deployment caused onboarding failures.",
  "summary": "The investigation examined whether Tuesday's deployment caused onboarding failures. Monitoring source A indicated a failure increase at 09:00, while monitoring source B reported the failure increase began at 12:10. Deployment records showed the deployment was completed at 11:30. The conflicting evidence regarding the timing of the failure increase and the deployment completion makes it impossible to establish a causal link between the deployment and the onboarding failures at this time.",
  "severity": "medium",
  "hypotheses": [
    "Hypothesis 1: The deployment caused the onboarding failures.",
    "Hypothesis 2: The failure increase is unrelated to the deployment."
  ],
  "evidence_needed": [
    "Confirmation of the exact timing of the failure increase.",
    "Further analysis to establish a causal link between the deployment and the failures."
  ],
  "recommended_actions": [
    "C

## Experiment 4 - Build tiny semantic gate

- 1st deterministic layer after Pydantic validation

#### For controlled test cases, we know a few facts ahead of time

- This is rule-based, using domain knowledge
- This is not a universal semantic validator.
- It is a deterministic test for a controlled experiment


In [31]:
@dataclass
class SemanticEvaluation:
    checks: dict[str, bool] = field(
        default_factory=dict
    )

    notes: list[str] = field(
        default_factory=list
    )

    @property
    def passed(self) -> bool:
        return all(self.checks.values())

def evaluate_semantics(
    brief: InvestigationBrief,
    *,
    expected_objective_terms: list[str],
    forbidden_terms: list[str],
    required_terms: list[str] | None = None,
) -> SemanticEvaluation:

    combined_text = " ".join(
        [
            brief.objective,
            brief.summary,
            *brief.hypotheses,
            *brief.evidence_needed,
            *brief.recommended_actions,
        ]
    ).lower()

    result = SemanticEvaluation()

    result.checks[
        "objective_preserved"
    ] = all(
        term.lower() in brief.objective.lower()
        for term in expected_objective_terms
    )

    result.checks[
        "forbidden_claims_absent"
    ] = all(
        term.lower() not in combined_text
        for term in forbidden_terms
    )

    if required_terms:
        result.checks[
            "required_evidence_preserved"
        ] = all(
            term.lower() in combined_text
            for term in required_terms
        )

    return result

In [33]:
b_semantics = evaluate_semantics(
    brief_b,
    expected_objective_terms=[
        "onboarding",
    ],
    forbidden_terms=[
        "tuesday's deployment",
        "deployment caused",
        "outage",
    ],
)

print("Schema valid:")
print(b_structure["schema_valid"])

print("\nSemantic checks:")
for name, passed in b_semantics.checks.items():
    print(
        f"{name:30s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

print()
print(
    "Overall semantic result:",
    "PASS"
    if b_semantics.passed
    else "FAIL",
)

Schema valid:
True

Semantic checks:
objective_preserved           : FAIL
forbidden_claims_absent       : FAIL

Overall semantic result: FAIL


In [34]:
c1_semantics = evaluate_semantics(
    brief_c1,
    expected_objective_terms=[
        "onboarding",
    ],
    forbidden_terms=[
        "payment gateway",
        "database corruption",
        "network outage",
        "deployment caused",
        "identity provider outage",
    ],
    required_terms=[
        "4%",
        "11%",
    ],
)

print(
    "Schema valid:",
    c1_structure["schema_valid"],
)

print()

for name, passed in c1_semantics.checks.items():
    print(
        f"{name:30s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

Schema valid: True

objective_preserved           : PASS
forbidden_claims_absent       : PASS
required_evidence_preserved   : PASS


In [35]:
rows = [
    {
        "experiment": "A constrained syntax",
        "json_valid": (
            a_constrained_validation["json_valid"]
        ),
        "schema_valid": (
            a_constrained_validation["schema_valid"]
        ),
        "semantic_valid": "not evaluated",
    },
    {
        "experiment": "B semantic sabotage",
        "json_valid": b_structure["json_valid"],
        "schema_valid": b_structure["schema_valid"],
        "semantic_valid": b_semantics.passed,
    },
    {
        "experiment": "C insufficient evidence",
        "json_valid": c1_structure["json_valid"],
        "schema_valid": c1_structure["schema_valid"],
        "semantic_valid": c1_semantics.passed,
    },
]

for row in rows:
    print(row)

{'experiment': 'A constrained syntax', 'json_valid': True, 'schema_valid': True, 'semantic_valid': 'not evaluated'}
{'experiment': 'B semantic sabotage', 'json_valid': True, 'schema_valid': True, 'semantic_valid': False}
{'experiment': 'C insufficient evidence', 'json_valid': True, 'schema_valid': True, 'semantic_valid': True}
